In [9]:
import ollama
from loguru import logger

In [10]:
llm_model = "gpt-oss:20b"

#### Define Tool

In [11]:
def wikipedia(country: str) -> str:
    """"
    Wikipedia tool function that simulates fetching information about a country from Wikipedia.

    Args:
        country (str): The name of the country to query.
    
    Returns:
        str: A string containing the information retrieved from Wikipedia about the specified country.
    """
    try:
        capitals = {
            'Italy': 'Rome',
            'Spain': 'Madrid',
        }
        capital = capitals.get(country, "Capital not found.")
        print(f"Fetched from Wikipedia: The capital of {country} is {capital}.")
        return capital
    except Exception as e:
        logger.error(f"Error calling Wikipedia API: {e}")
        return "Error fetching data from Wikipedia."

### ReAct (Reason + Act)
ReAct is the most fundamental agentic design pattern. If you only learn one pattern, make it this one.

#### The idea
ReAct combines reasoning (thinking about what to do) with acting (actually doing it) in an interleaved loop. The agent:

* **Thinks** about the current situation
* **Acts** by calling a tool or taking a step
* **Observes** the result gathered

Repeats until the task is done

In [ ]:
class ReActAgent:
    
    def __init__(self, system="", max_iterations=20):
        self.system = system
        self.full_memory = []
        self.tools_used = []
        self.max_iterations = max_iterations
        if self.system:
            self.full_memory.append({"role": "system", "content": system})

    def __call__(self, message):
        try:
            self.full_memory.append({"role": "user", "content": message})
            result = self.execute()
            self.full_memory.append({"role": "assistant", "content": result})
            return result
        except Exception as e:
            logger.error(f"Error during agent call: {e}")
            return "An error occurred while processing your request."

    def execute(self):
        iterations = 0

        while iterations < self.max_iterations:
            iterations += 1
            logger.info(f"[ReAct] Iteration {iterations}/{self.max_iterations}")

            # --- THINK ---
            # The model reasons about what to do next
            response = ollama.chat(
                model=llm_model,
                messages=self.full_memory,
                options={"temperature": 0},
                tools=[wikipedia]
            )

            message = response["message"]
            tool_calls = message.get("tool_calls", [])

            # --- ACT ---
            # The model decides to call tools or provide a final answer
            if tool_calls:

                self.full_memory.append({
                    "role": "assistant",
                    "content": message.get("content", ""),
                    "tool_calls": tool_calls
                })

                logger.info(f"[ReAct] Model decided to call {len(tool_calls)} tool(s).")

                for call in tool_calls:
                    tool_name = call.function.name
                    tool_args = call.function.arguments
                    logger.info(f"[ReAct] ACT  → calling '{tool_name}' with args {tool_args}")

                    # --- OBSERVE ---
                    # Run the tool and feed the result back
                    if tool_name == "wikipedia":
                        observation = wikipedia(tool_args["country"])
                    else:
                        observation = f"Unknown tool: {tool_name}"
                        logger.warning(f"[ReAct] Unknown tool requested: {tool_name}")

                    logger.info(f"[ReAct] OBSERVE → {str(observation)[:200]}...")
                    self.tools_used.append(tool_name)

                    self.full_memory.append({
                        "role": "tool",
                        "tool_name": tool_name,
                        "content": str(observation)
                    })

            else:
                # if no tool calls, model provides the final answer 
                final_answer = message.get("content", "")
                logger.info(f"[ReAct] Final answer reached after {iterations} iteration(s).")
                return final_answer

        # max iterations hit without a conclusive answer
        logger.warning(f"[ReAct] Max iterations ({self.MAX_ITERATIONS}) reached without final answer.")
        return "I was unable to reach a final answer within the allowed number of steps."

In [ ]:
react_agent = ReActAgent(system="You are a helpful assistant that can use tools to answer questions.")
response = react_agent("What is the capital of Italy? Look at Wikipedia if you don't know.")

2026-05-03 15:09:47.274 | INFO     | __main__:execute:27 - [ReAct] Iteration 1/10
2026-05-03 15:09:49.849 | INFO     | __main__:execute:51 - [ReAct] Model decided to call 1 tool(s).
2026-05-03 15:09:49.850 | INFO     | __main__:execute:56 - [ReAct] ACT  → calling 'wikipedia' with args {'country': 'Italy'}
2026-05-03 15:09:49.851 | INFO     | __main__:execute:66 - [ReAct] OBSERVE → Rome...
2026-05-03 15:09:49.851 | INFO     | __main__:execute:27 - [ReAct] Iteration 2/10


Fetched from Wikipedia: The capital of Italy is Rome.


2026-05-03 15:09:52.541 | INFO     | __main__:execute:78 - [ReAct] Final answer reached after 2 iteration(s).
